In [1]:
import sys
sys.path.append("E:\\wikiart_project")

from config import (
    set_seed, get_split_indices, get_transforms,
    WikiArtDataset, LEARNING_RATE, NUM_STYLES, SAVE_DIR
)

import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from datasets import load_dataset
from tqdm import tqdm
import matplotlib.pyplot as plt

# ── Set seed ─────────────────────────────────────────────
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Settings (3 epoch only, single task) ────────────────
NUM_EPOCHS = 3
IMAGE_SIZE = 224
BATCH_SIZE = 32
MODEL_NAME = "resnet50_single_task"

# ── Load dataset ────────────────────────────────────────
print("Loading dataset...")
dataset = load_dataset("huggan/wikiart", split="train")
splits = get_split_indices(len(dataset))
print(f"Train: {len(splits['train'])} | Val: {len(splits['val'])}")

train_transform, eval_transform = get_transforms(IMAGE_SIZE)

train_dataset = WikiArtDataset(dataset, splits['train'], train_transform)
val_dataset   = WikiArtDataset(dataset, splits['val'],   eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ── Single-task ResNet-50 (style only) ──────────────────
class SingleTaskResNet(nn.Module):
    def __init__(self, num_styles=NUM_STYLES):
        super().__init__()
        backbone = models.resnet50(weights='IMAGENET1K_V1')
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        # Only one head: style
        self.style_head = nn.Linear(2048, num_styles)

    def forward(self, x):
        f = self.backbone(x).flatten(1)
        return self.style_head(f)

model = SingleTaskResNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler()

# ── Checkpoint resume ───────────────────────────────────
CHECKPOINT_PATH = f"{SAVE_DIR}\\{MODEL_NAME}_checkpoint.pth"
start_epoch = 0
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

if os.path.exists(CHECKPOINT_PATH):
    print("Found checkpoint, resuming...")
    ckpt = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scaler.load_state_dict(ckpt['scaler_state'])
    start_epoch = ckpt['epoch'] + 1
    train_losses = ckpt['train_losses']
    val_losses   = ckpt['val_losses']
    train_accs   = ckpt['train_accs']
    val_accs     = ckpt['val_accs']
    print(f"Resumed from epoch {start_epoch}")

print("Model ready!")

# ── Training loop (only style loss) ─────────────────────
for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, styles, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"):
        images, styles = images.to(device), styles.to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            style_out = model(images)
            loss = criterion(style_out, styles)  # ONLY style loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        correct      += (style_out.argmax(1) == styles).sum().item()
        total        += styles.size(0)

    train_losses.append(running_loss / len(train_loader))
    train_accs.append(correct / total)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, styles, _ in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            images, styles = images.to(device), styles.to(device)
            with torch.cuda.amp.autocast():
                style_out = model(images)
                loss = criterion(style_out, styles)
            val_loss    += loss.item()
            val_correct += (style_out.argmax(1) == styles).sum().item()
            val_total   += styles.size(0)

    val_losses.append(val_loss / len(val_loader))
    val_accs.append(val_correct / val_total)

    print(f"Epoch {epoch+1}: Train Acc={train_accs[-1]:.4f} | Val Acc={val_accs[-1]:.4f}")

    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state': scaler.state_dict(),
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs,
    }, CHECKPOINT_PATH)

torch.save(model.state_dict(), f"{SAVE_DIR}\\{MODEL_NAME}_wikiart.pth")
history = {'train_losses': train_losses, 'val_losses': val_losses,
           'train_accs': train_accs, 'val_accs': val_accs}
with open(f"{SAVE_DIR}\\{MODEL_NAME}_history.json", "w") as f:
    json.dump(history, f)

print("\n" + "="*60)
print("ABLATION 1 RESULT: Single-Task vs Multi-Task")
print("="*60)
print(f"Single-task ResNet-50 (3 epoch): Val Acc = {val_accs[-1]:.4f}")
print(f"Multi-task  ResNet-50 (5 epoch): Val Acc ≈ 0.5916")
print("(Note: We compare epoch 3 single-task vs epoch 3 multi-task for fairness)")
print("All done!")

Using device: cuda
Loading dataset...


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/45 [00:00<?, ?it/s]

Train: 57010 | Val: 12217


C:\Users\yimin\AppData\Local\Temp\ipykernel_26912\778610122.py:61: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


...

Epoch 1: Train Acc=0.4772 | Val Acc=0.5477


Epoch 2/3 [Val]: 100%|██████████| 382/382 [05:16<00:00,  1.21it/s]


Epoch 2: Train Acc=0.5815 | Val Acc=0.5828


Epoch 3/3 [Val]: 100%|██████████| 382/382 [05:17<00:00,  1.20it/s]


Epoch 3: Train Acc=0.6436 | Val Acc=0.5847

ABLATION 1 RESULT: Single-Task vs Multi-Task
Single-task ResNet-50 (3 epoch): Val Acc = 0.5847
Multi-task  ResNet-50 (5 epoch): Val Acc ≈ 0.5916
(Note: We compare epoch 3 single-task vs epoch 3 multi-task for fairness)
All done!


In [2]:
import json

with open(f"E:\\wikiart_project\\resnet50_history.json", "r") as f:
    multi_task_history = json.load(f)

print("Multi-task ResNet-50 history:")
for i, (tr, vl) in enumerate(zip(multi_task_history['train_accs'], multi_task_history['val_accs'])):
    print(f"Epoch {i+1}: Train Acc={tr:.4f} | Val Acc={vl:.4f}")

Multi-task ResNet-50 history:
Epoch 1: Train Acc=0.4805 | Val Acc=0.5648
Epoch 2: Train Acc=0.5892 | Val Acc=0.5739
Epoch 3: Train Acc=0.6534 | Val Acc=0.6007
Epoch 4: Train Acc=0.7076 | Val Acc=0.5976
Epoch 5: Train Acc=0.7549 | Val Acc=0.5916
